# Set WD to Root

In [1]:
from pathlib import Path
import os

repo_root = Path.cwd().parents[0]
os.chdir(repo_root)

print(Path.cwd())

d:\1_Document\A-02_Notebooks\1_UWA_Learning_Archive\02_Units\Y1S2_CITS4012 - Natural Language Processing\Assignment\Assignment_02\repo


# Check Repeat Goals

In [2]:
import pandas as pd
from pathlib import Path
from scripts.data_loader.load_piqa_data import piqa_to_dataframe

# 1. 加载原始划分的数据集
BASE_PATH = Path('./datasets/PIQA')
train_df, valid_df, _ = piqa_to_dataframe(base_path=BASE_PATH)

print(f"训练集行数: {len(train_df)}")
print(f"验证集行数: {len(valid_df)}")

# ---------------------------------------------------------
# 验证 1: 检查 Goal 重叠率 (Row-Split 泄露量化)
# ---------------------------------------------------------
train_goals = set(train_df['goal'].dropna().unique())
valid_goals = valid_df['goal'].dropna()

# 找出验证集中，Goal 已经在训练集中出现过的行
leaked_mask = valid_goals.isin(train_goals)
leaked_count = leaked_mask.sum()
total_valid = len(valid_df)
leak_ratio = (leaked_count / total_valid) * 100

print("\n=== 1. Goal 重叠泄露验证 ===")
print(f"验证集中共享训练集 Goal 的行数: {leaked_count} / {total_valid}")
print(f"泄露比例: {leak_ratio:.2f}%")

# ---------------------------------------------------------
# 验证 2: 进一步分析重叠行的正确选项/错误选项差异
# ---------------------------------------------------------
print("\n=== 2. 重叠行具体内容抽样检查 ===")

# 提取出验证集中被“泄露”的样本
leaked_valid_samples = valid_df[leaked_mask]

# 随机抽查 3 个重叠的 Goal，对比它在训练集和验证集里的内容
sample_goals = leaked_valid_samples['goal'].drop_duplicates().head(3)

for idx, goal in enumerate(sample_goals, 1):
    print(f"\n--- [案例 {idx}] Goal: \"{goal}\" ---")
    
    # 获取训练集中该 Goal 的所有记录
    train_matches = train_df[train_df['goal'] == goal]
    print("【训练集里的记录】:")
    for _, row in train_matches.iterrows():
        # 假设 label 为 0 表示 sol1，1 表示 sol2（根据你数据集的具体字段调整）
        correct_sol = row['sol1'] if row['label'] == 0 else row['sol2']
        wrong_sol = row['sol2'] if row['label'] == 0 else row['sol1']
        print(f"  - 正确项: {correct_sol} | 干扰项: {wrong_sol}")
        
    # 获取验证集中该 Goal 的所有记录
    valid_matches = valid_df[valid_df['goal'] == goal]
    print("【验证集里的记录】:")
    for _, row in valid_matches.iterrows():
        correct_sol = row['sol1'] if row['label'] == 0 else row['sol2']
        wrong_sol = row['sol2'] if row['label'] == 0 else row['sol1']
        print(f"  - 正确项: {correct_sol} | 干扰项: {wrong_sol}")

训练集行数: 14502
验证集行数: 1611

=== 1. Goal 重叠泄露验证 ===
验证集中共享训练集 Goal 的行数: 157 / 1611
泄露比例: 9.75%

=== 2. 重叠行具体内容抽样检查 ===

--- [案例 1] Goal: "bin" ---
【训练集里的记录】:
  - 正确项: can crush a tablet into powder | 干扰项: can crush a shirt into powder
  - 正确项: can hold wet octopus parts easily | 干扰项: can hold negative vacuum easily
  - 正确项: can hold a comb on it's edges | 干扰项: can hold a brush on it's edges
  - 正确项: can be used as a home for bunnies | 干扰项: can be used as a food for bunnies
  - 正确项: can contain tank of gas | 干扰项: can contain waste of gas
【验证集里的记录】:
  - 正确项: can hold books | 干扰项: can hold car

--- [案例 2] Goal: "chopper" ---
【训练集里的记录】:
  - 正确项: flying mountains easily | 干扰项: flying crowd easily
  - 正确项: can carry plastic toys | 干扰项: can carry cotton ball
  - 正确项: can be used to cut fruit that are thick | 干扰项: can be used to cut wires that are thick
【验证集里的记录】:
  - 正确项: chop vegetables easily. | 干扰项: chop grass easily.

--- [案例 3] Goal: "pots and pans" ---
【训练集里的记录】:
  - 正确项: can hurt the foot

# Check New Duplicate for Updated Pipeline

In [2]:
import pandas as pd
from pathlib import Path
from scripts.data_loader.load_piqa_data import piqa_to_dataframe

# 1. 加载原始划分的数据集
BASE_PATH = Path('./datasets/PIQA')
train_df, valid_df, _ = piqa_to_dataframe(base_path=BASE_PATH)

print(f"训练集行数: {len(train_df)}")
print(f"验证集行数: {len(valid_df)}")

# ---------------------------------------------------------
# 验证 1: 检查 Goal 重叠率 (Row-Split 泄露量化)
# ---------------------------------------------------------
train_goals = set(train_df['goal'].dropna().unique())
valid_goals = valid_df['goal'].dropna()

# 找出验证集中，Goal 已经在训练集中出现过的行
leaked_mask = valid_goals.isin(train_goals)
leaked_count = leaked_mask.sum()
total_valid = len(valid_df)
leak_ratio = (leaked_count / total_valid) * 100

print("\n=== 1. Goal 重叠泄露验证 ===")
print(f"验证集中共享训练集 Goal 的行数: {leaked_count} / {total_valid}")
print(f"泄露比例: {leak_ratio:.2f}%")

# ---------------------------------------------------------
# 验证 2: 进一步分析重叠行的正确选项/错误选项差异
# ---------------------------------------------------------
print("\n=== 2. 重叠行具体内容抽样检查 ===")

# 提取出验证集中被“泄露”的样本
leaked_valid_samples = valid_df[leaked_mask]

# 随机抽查 3 个重叠的 Goal，对比它在训练集和验证集里的内容
sample_goals = leaked_valid_samples['goal'].drop_duplicates().head(3)

for idx, goal in enumerate(sample_goals, 1):
    print(f"\n--- [案例 {idx}] Goal: \"{goal}\" ---")
    
    # 获取训练集中该 Goal 的所有记录
    train_matches = train_df[train_df['goal'] == goal]
    print("【训练集里的记录】:")
    for _, row in train_matches.iterrows():
        # 假设 label 为 0 表示 sol1，1 表示 sol2（根据你数据集的具体字段调整）
        correct_sol = row['sol1'] if row['label'] == 0 else row['sol2']
        wrong_sol = row['sol2'] if row['label'] == 0 else row['sol1']
        print(f"  - 正确项: {correct_sol} | 干扰项: {wrong_sol}")
        
    # 获取验证集中该 Goal 的所有记录
    valid_matches = valid_df[valid_df['goal'] == goal]
    print("【验证集里的记录】:")
    for _, row in valid_matches.iterrows():
        correct_sol = row['sol1'] if row['label'] == 0 else row['sol2']
        wrong_sol = row['sol2'] if row['label'] == 0 else row['sol1']
        print(f"  - 正确项: {correct_sol} | 干扰项: {wrong_sol}")

训练集行数: 14507
验证集行数: 1606

=== 1. Goal 重叠泄露验证 ===
验证集中共享训练集 Goal 的行数: 0 / 1606
泄露比例: 0.00%

=== 2. 重叠行具体内容抽样检查 ===
